# Exercises — Chapter 08: Domain-Specific Financial LLMs

Complete the exercises from Lecture 08.

In [ ]:
# Your code here

## Data Lab — SEC EDGAR

Analyse the vocabulary of financial filings to understand what makes finance text different from general English — and why domain-adapted language models outperform general-purpose ones.

In [ ]:
import requests, time, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

EDGAR_HEADERS = {"User-Agent": "LLM-Finance-Course instructor@dauphine.eu"}

def edgar_get(url):
    time.sleep(0.11)
    r = requests.get(url, headers=EDGAR_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

def get_cik(ticker):
    data = edgar_get("https://www.sec.gov/files/company_tickers.json")
    for v in data.values():
        if v["ticker"].upper() == ticker.upper():
            return str(v["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found")

def get_submissions(cik):
    return edgar_get(f"https://data.sec.gov/submissions/CIK{cik}.json")

def get_concept(cik, concept):
    return edgar_get(
        f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json")

def get_annual_series(cik, concept):
    data = get_concept(cik, concept)
    usd  = data.get("units", {}).get("USD", [])
    rows = [u for u in usd if u.get("form") == "10-K" and u.get("filed")]
    if not rows:
        for _, vals in data.get("units", {}).items():
            rows = [u for u in vals if u.get("form") == "10-K" and u.get("filed")]
            if rows: break
    df = (pd.DataFrame(rows)[["end","val","filed","accn"]]
            .rename(columns={"end":"date","val":concept})
            .drop_duplicates("date").sort_values("date"))
    df["date"] = pd.to_datetime(df["date"])
    return df

def fetch_10k_html(ticker):
    cik  = get_cik(ticker)
    subs = get_submissions(cik)
    f    = subs["filings"]["recent"]
    idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
    acc  = f["accessionNumber"][idx].replace("-", "")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}"
            f"/{acc}/{f['primaryDocument'][idx]}")
    time.sleep(0.15)
    return requests.get(url, headers=EDGAR_HEADERS, timeout=30).text

def extract_mda(html):
    text = re.sub(r"<[^>]+>", " ", html)
    m    = re.search(
        r"Item\s+7[.\s]+Management.{0,80}Discussion.*?(?=Item\s+7A|Item\s+8)",
        text, re.IGNORECASE | re.DOTALL)
    raw  = m.group(0) if m else text[:30000]
    return re.sub(r"\s+", " ", raw).strip()

### Exercise [B]: Finance-Domain Vocabulary Size

In [ ]:
# --- Exercise [B]: Finance-Domain Vocabulary Size ---
TICKERS_08 = ["AAPL", "JPM", "XOM"]
vocab_data = {}
for t in TICKERS_08:
    html = fetch_10k_html(t)
    mda  = extract_mda(html)
    toks = re.findall(r"[a-z]{2,}", mda.lower())
    from collections import Counter
    freq  = Counter(toks)
    hapax = sum(1 for v in freq.values() if v == 1)
    vocab_data[t] = {
        "total": len(toks),
        "unique": len(freq),
        "density": len(freq)/max(len(toks),1),
        "hapax_frac": hapax/max(len(freq),1),
    }
    print(f"{t}: {len(toks):,} tokens, {len(freq):,} unique, {hapax} hapax")

labels = list(vocab_data.keys())
metrics = ["total","unique","density","hapax_frac"]
fig, axes = plt.subplots(1, 2, figsize=(11,4))
for ax, m in zip(axes, ["density","hapax_frac"]):
    vals = [vocab_data[t][m] for t in labels]
    ax.bar(labels, vals, color=["steelblue","tomato","goldenrod"])
    ax.set_title(m.replace("_"," ").title()); ax.set_ylabel("Fraction")
plt.tight_layout(); plt.show()

### Exercise [I]: Out-of-Vocabulary Rate vs. General English

In [ ]:
# --- Exercise [I]: OOV Rate ---
COMMON_500 = {
    "the","be","to","of","and","a","in","that","have","it","for","not","on","with",
    "he","as","you","do","at","this","but","his","by","from","they","we","say","her",
    "she","or","an","will","my","one","all","would","there","their","what","so","up",
    "out","if","about","who","get","which","go","me","when","make","can","like","time",
    "no","just","him","know","take","people","into","year","your","good","some","could",
    "them","see","other","than","then","now","look","only","come","its","over","think",
    "also","back","after","use","two","how","our","work","first","well","way","even",
    "new","want","because","any","these","give","day","most","us","between","need","large",
    "often","within","may","more","such","both","each","been","has","had","were","are",
    "was","is","said","company","net","total","fiscal","year","million","billion",
    "financial","revenue","income","operating","growth","increased","decreased","per",
    "quarter","results","diluted","share","shares","basis","points","cost","business",
    "services","products","market","customers","segment","period","compared"
}

from collections import Counter

oov_data = {}
for t in TICKERS_08:
    html = fetch_10k_html(t)
    mda  = extract_mda(html)
    toks = re.findall(r"[a-z]{2,}", mda.lower())
    freq  = Counter(toks)
    oov_toks = {w for w in freq if w not in COMMON_500}
    oov_rate  = len(oov_toks) / max(len(freq),1)
    top20_oov = sorted(oov_toks, key=lambda w: -freq[w])[:20]
    oov_data[t] = {"oov_rate": oov_rate, "top20": top20_oov}
    print(f"{t}: OOV rate={oov_rate:.3f}")
    print(f"  Top OOV: {', '.join(top20_oov[:10])}")

fig, ax = plt.subplots(figsize=(6,3))
ax.barh(list(oov_data.keys()), [v["oov_rate"] for v in oov_data.values()], color="steelblue")
ax.set_xlabel("OOV Rate"); ax.set_title("Fraction of MD&A Vocab Not in Top-500 English Words")
plt.tight_layout(); plt.show()

### Exercise [A]: Domain Vocabulary Drift

In [ ]:
# --- Exercise [A]: Domain Vocabulary Drift ---
cik_msft = get_cik("MSFT")
subs_m = get_submissions(cik_msft)
fm     = subs_m["filings"]["recent"]
k10_idx = [i for i, x in enumerate(fm["form"]) if x == "10-K"][:5]

year_vocabs = {}
for i in k10_idx:
    year = int(fm["filingDate"][i][:4])
    acc  = fm["accessionNumber"][i].replace("-","")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik_msft.lstrip('0')}"
            f"/{acc}/{fm['primaryDocument'][i]}")
    time.sleep(0.15)
    raw  = requests.get(url, headers=EDGAR_HEADERS, timeout=30).text
    mda  = extract_mda(raw)
    year_vocabs[year] = set(re.findall(r"[a-z]{3,}", mda.lower()))
    print(f"{year}: {len(year_vocabs[year]):,} unique tokens")

years_sorted = sorted(year_vocabs.keys())
jaccard_pairs = []
for i in range(len(years_sorted)-1):
    ya, yb = years_sorted[i], years_sorted[i+1]
    va, vb = year_vocabs[ya], year_vocabs[yb]
    j = len(va & vb) / len(va | vb) if (va | vb) else 0
    jaccard_pairs.append((f"{ya}\u2192{yb}", j))

labels_j, vals_j = zip(*jaccard_pairs)
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(labels_j, vals_j, "o-", color="steelblue", linewidth=2)
ax.set_ylabel("Jaccard Similarity"); ax.set_ylim(0, 1)
ax.set_title("MSFT MD&A Vocabulary: Year-on-Year Jaccard Similarity")
ax.tick_params(axis="x", rotation=15); plt.tight_layout(); plt.show()

min_pair = min(jaccard_pairs, key=lambda x: x[1])
print(f"Lowest similarity: {min_pair[0]} (J={min_pair[1]:.3f})")